In [4]:
%pip install lxml

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached lxml-6.1.3-cp312-cp312-win_amd64.whl.metadata (3.4 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:

import os
import pandas as pd
from datetime import datetime, timezone

# =====================================
# 1. OUTPUT FOLDER
# =====================================

OUTPUT_DIR = "phone_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SOURCE = (
    "https://www.whoseno.com/blog/"
    "most-reported-spam-numbers-pakistan/"
)

# =====================================
# 2. WHOSENO PUBLISHED TOP 10 DATA
# =====================================

records = [
    ["0517080562", 553, 381, 68.9],
    ["0512521013", 385, 232, 60.3],
    ["0512521025", 296, 221, 74.7],
    ["03481117373", 275, 186, 67.6],
    ["03135442150", 261, 172, 65.9],
    ["0512521017", 221, 145, 65.6],
    ["0518857373", 145, 89, 61.4],
    ["0512521019", 130, 78, 60.0],
    ["03443737111", 122, 78, 63.9],
    ["03135442340", 121, 73, 60.3]
]

spam_df = pd.DataFrame(
    records,
    columns=[
        "phone_number",
        "total_reports",
        "explicit_negative",
        "negative_share"
    ]
)

spam_df["label"] = "spam"
spam_df["label_basis"] = "WhoseNo community reports"
spam_df["source"] = SOURCE
spam_df["scraped_at"] = datetime.now(
    timezone.utc
).isoformat()

# =====================================
# 3. SAVE SPAM CSV
# =====================================

spam_path = os.path.join(
    OUTPUT_DIR,
    "spam_numbers.csv"
)

spam_df.to_csv(
    spam_path,
    index=False,
    encoding="utf-8-sig"
)

print("Spam CSV created:", spam_path)
print("Spam records:", len(spam_df))

# =====================================
# 4. LOAD HAM CSV
# =====================================

ham_path = os.path.join(
    OUTPUT_DIR,
    "ham_numbers.csv"
)

if not os.path.exists(ham_path):

    pd.DataFrame(
        columns=["phone_number", "source"]
    ).to_csv(
        ham_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Ham template created:", ham_path)

ham_df = pd.read_csv(
    ham_path,
    dtype=str
)

if "phone_number" not in ham_df.columns:
    raise ValueError(
        "ham_numbers.csv needs phone_number column"
    )

# Clean and validate empty rows
ham_df["phone_number"] = (
    ham_df["phone_number"]
    .fillna("")
    .str.strip()
)

ham_df = ham_df[
    ham_df["phone_number"] != ""
].copy()

ham_df["label"] = "ham"
ham_df["label_basis"] = "User-provided verified source"

if "source" not in ham_df.columns:
    ham_df["source"] = "User-provided"

ham_df = ham_df[
    [
        "phone_number",
        "label",
        "label_basis",
        "source"
    ]
]

# =====================================
# 5. PREPARE SPAM DATA
# =====================================

spam_records = spam_df[
    [
        "phone_number",
        "label",
        "label_basis",
        "source"
    ]
].copy()

# =====================================
# 6. MERGE DATASETS
# =====================================

combined_df = pd.concat(
    [spam_records, ham_df],
    ignore_index=True
)

combined_df = combined_df.drop_duplicates(
    subset=["phone_number"],
    keep="first"
)

final_path = os.path.join(
    OUTPUT_DIR,
    "phone_numbers_dataset.csv"
)

combined_df.to_csv(
    final_path,
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 7. DISPLAY RESULTS
# =====================================

print("\n========== SUCCESS ==========")

print("Spam records:", len(spam_records))
print("Ham records:", len(ham_df))
print("Total unique records:", len(combined_df))

print("\nLabel distribution:")
print(combined_df["label"].value_counts())

print("\nFinal dataset:")
print(combined_df.head(10).to_string(index=False))

print("\nFiles saved:")
print(spam_path)
print(ham_path)
print(final_path)

Spam CSV created: phone_dataset\spam_numbers.csv
Spam records: 10
Ham template created: phone_dataset\ham_numbers.csv

========== SUCCESS ==========
Spam records: 10
Ham records: 0
Total unique records: 10

Label distribution:
label
spam    10
Name: count, dtype: int64

Final dataset:
phone_number label               label_basis                                                            source
  0517080562  spam WhoseNo community reports https://www.whoseno.com/blog/most-reported-spam-numbers-pakistan/
  0512521013  spam WhoseNo community reports https://www.whoseno.com/blog/most-reported-spam-numbers-pakistan/
  0512521025  spam WhoseNo community reports https://www.whoseno.com/blog/most-reported-spam-numbers-pakistan/
 03481117373  spam WhoseNo community reports https://www.whoseno.com/blog/most-reported-spam-numbers-pakistan/
 03135442150  spam WhoseNo community reports https://www.whoseno.com/blog/most-reported-spam-numbers-pakistan/
  0512521017  spam WhoseNo community reports htt